- 왼쪽 카메라 이미지 `*_left.png`
- 오른쪽 카메라 이미지 `*_right.png`
- 시차(disparity) 이미지 (`*_disp.png` 또는 `*_disp16.png`)
- confidnece 이미지 (`*_confidence.png` 등)

In [ ]:
import os
import re
import glob
import math
import configparser
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, RANSACRegressor


# -----------------------------
# 1) CONF 파싱 & 해상도별 intrinsics 선택
# -----------------------------
@dataclass
class Intrinsics:
    fx: float
    fy: float
    cx: float
    cy: float


@dataclass
class StereoConfig:
    baseline_m: float
    intr_left: Intrinsics


def parse_conf(conf_path: str) -> dict:
    """
    Depth_001.conf 같은 INI 형태를 dict로 읽는다.
    """
    cp = configparser.ConfigParser()
    cp.read(conf_path, encoding="utf-8")
    data = {}
    for sec in cp.sections():
        data[sec] = {k: float(v) for k, v in cp.items(sec)}
    return data


def pick_intrinsics_by_image_size(conf_dict: dict, width: int, height: int) -> Intrinsics:
    """
    이미지 해상도 보고 LEFT_CAM_{2K|FHD|HD|VGA} 중 선택.
    (너희 데이터 해상도에 맞게 매핑만 수정하면 됨)
    """
    # 대략적인 매핑 (필요 시 조정)
    if width >= 2000:  # 2K 근처
        sec = "LEFT_CAM_2K"
    elif width >= 1800:  # 1920x1080
        sec = "LEFT_CAM_FHD"
    elif width >= 1200:  # 1280x720
        sec = "LEFT_CAM_HD"
    else:
        sec = "LEFT_CAM_VGA"

    intr = conf_dict[sec]
    return Intrinsics(fx=intr["fx"], fy=intr["fy"], cx=intr["cx"], cy=intr["cy"])


def load_stereo_config(conf_path: str, sample_image_path: str) -> StereoConfig:
    conf = parse_conf(conf_path)

    # baseline: 보통 mm -> m 변환 가정
    baseline_mm = conf["STEREO"]["baseline"]
    baseline_m = baseline_mm / 1000.0

    # 이미지 크기로 intrinsics 선택
    img = cv2.imread(sample_image_path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot read sample image: {sample_image_path}")
    h, w = img.shape[:2]
    intr = pick_intrinsics_by_image_size(conf, w, h)
    return StereoConfig(baseline_m=baseline_m, intr_left=intr)


# -----------------------------
# 2) disparity -> depth
# -----------------------------
def read_disp16(path: str) -> np.ndarray:
    """
    disp16.png는 16-bit일 가능성이 높다.
    IMREAD_UNCHANGED로 읽으면 uint16 유지.
    """
    disp = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if disp is None:
        raise FileNotFoundError(f"Cannot read disparity: {path}")
    if disp.dtype != np.uint16 and disp.dtype != np.int16:
        # 가끔 uint8로 들어올 수도 있으니 방어
        disp = disp.astype(np.uint16)
    return disp


def guess_disp_scale(disp16: np.ndarray) -> float:
    """
    disparity 스케일 추정 (휴리스틱).
    ZED 계열은 raw가 fixed-point로 저장되는 경우가 많아서 /16을 자주 씀.
    여기서는 우선 후보를 반환하고, depth sanity check로 고르도록 한다.
    """
    # 후보들 (필요하면 더 추가)
    return 16.0


def disparity_to_depth_m(disp_raw: np.ndarray, fx: float, baseline_m: float, scale: float) -> np.ndarray:
    """
    disp_raw: uint16
    scale: raw를 실제 disparity(px)로 만들기 위한 분모 (예: 16)
    """
    disp = disp_raw.astype(np.float32) / float(scale)

    # 0 또는 너무 작은 disparity는 제외 (무한대 depth 방지)
    disp = np.where(disp > 0.1, disp, np.nan)

    depth = (fx * baseline_m) / disp  # meters
    return depth


# -----------------------------
# 3) ROI 포인트클라우드 만들기 + plane fit + slope
# -----------------------------
def build_points_from_depth(depth_m: np.ndarray, intr: Intrinsics, roi_y0: int) -> np.ndarray:
    """
    depth_m: (H, W) float32 with NaNs
    roi_y0: ROI 시작 y (예: int(H*0.6))
    반환: (N,3) points in camera coordinates (X,Y,Z)
    """
    H, W = depth_m.shape
    ys, xs = np.mgrid[roi_y0:H, 0:W]
    Z = depth_m[roi_y0:H, :]

    valid = np.isfinite(Z)
    xs = xs[valid].astype(np.float32)
    ys = ys[valid].astype(np.float32)
    Z = Z[valid].astype(np.float32)

    X = (xs - intr.cx) * Z / intr.fx
    Y = (ys - intr.cy) * Z / intr.fy

    pts = np.stack([X, Y, Z], axis=1)
    return pts


def fit_ground_plane_ransac(points_xyz: np.ndarray):
    """
    Y ≈ a*X + b*Z + c 로 평면 근사 (RANSAC)
    returns: (a,b,c), inlier_ratio
    """
    if points_xyz.shape[0] < 2000:
        return None, 0.0

    XZ = points_xyz[:, [0, 2]]  # [X, Z]
    Y = points_xyz[:, 1]

    base = LinearRegression()
    ransac = RANSACRegressor(
        estimator=base,
        min_samples=0.5,
        residual_threshold=0.02,  # 2cm 정도 (데이터/노이즈에 맞게 조정)
        max_trials=100,
        random_state=42
    )
    ransac.fit(XZ, Y)

    inliers = ransac.inlier_mask_
    inlier_ratio = float(np.mean(inliers))

    a = float(ransac.estimator_.coef_[0])
    b = float(ransac.estimator_.coef_[1])
    c = float(ransac.estimator_.intercept_)
    return (a, b, c), inlier_ratio


def compute_slope_from_plane_coeff(b: float) -> dict:
    """
    forward slope: dY/dZ = b
    slope% ≈ |b|*100
    angle = atan(|b|)
    """
    slope = abs(b) * 100.0
    angle_deg = math.degrees(math.atan(abs(b)))
    return {"slope_percent": slope, "slope_angle_deg": angle_deg}


# -----------------------------
# 4) 전체 실행 (폴더 단위)
# -----------------------------
def extract_frame_id(path: str) -> str:
    # ZED1_KSC_001032_disp16.png -> ZED1_KSC_001032
    base = os.path.basename(path)
    m = re.match(r"(.+)_disp16\.png$", base)
    return m.group(1) if m else os.path.splitext(base)[0]


def run_phase1(dataset_dir: str, conf_name: str = "Depth_001.conf", roi_ratio: float = 0.6):
    conf_path = os.path.join(dataset_dir, conf_name)
    disp_paths = sorted(glob.glob(os.path.join(dataset_dir, "*_disp16.png")))
    if not disp_paths:
        raise FileNotFoundError(f"No disp16 found in: {dataset_dir}")

    # intrinsics 선택을 위해 샘플 RGB(Left or right) 하나 고르기
    # left.png가 있으면 그걸 우선 사용
    sample_rgb = None
    for p in sorted(glob.glob(os.path.join(dataset_dir, "*_left.png"))):
        sample_rgb = p
        break
    if sample_rgb is None:
        # 대체: right.png
        for p in sorted(glob.glob(os.path.join(dataset_dir, "*_right.png"))):
            sample_rgb = p
            break
    if sample_rgb is None:
        raise FileNotFoundError("No *_left.png or *_right.png found to infer resolution/intrinsics.")

    stereo = load_stereo_config(conf_path, sample_rgb)
    intr = stereo.intr_left

    rows = []
    for disp_path in disp_paths:
        frame_id = extract_frame_id(disp_path)
        disp_raw = read_disp16(disp_path)

        H, W = disp_raw.shape[:2]
        roi_y0 = int(H * roi_ratio)

        # disp scale 추정 (기본 16.0 가정)
        scale = guess_disp_scale(disp_raw)

        depth = disparity_to_depth_m(disp_raw, fx=intr.fx, baseline_m=stereo.baseline_m, scale=scale)

        # depth sanity filter
        depth = np.where((depth > 0.3) & (depth < 50.0), depth, np.nan)

        pts = build_points_from_depth(depth, intr=intr, roi_y0=roi_y0)
        coeff, inlier_ratio = fit_ground_plane_ransac(pts)

        if coeff is None:
            rows.append({
                "frame_id": frame_id,
                "disp_path": disp_path,
                "H": H, "W": W,
                "disp_scale": scale,
                "inlier_ratio": inlier_ratio,
                "slope_percent": np.nan,
                "slope_angle_deg": np.nan,
                "status": "too_few_points"
            })
            continue

        a, b, c = coeff
        slope_info = compute_slope_from_plane_coeff(b)

        rows.append({
            "frame_id": frame_id,
            "disp_path": disp_path,
            "H": H, "W": W,
            "disp_scale": scale,
            "baseline_m": stereo.baseline_m,
            "fx": intr.fx, "fy": intr.fy, "cx": intr.cx, "cy": intr.cy,
            "plane_a_dYdX": a,
            "plane_b_dYdZ": b,
            "plane_c": c,
            "inlier_ratio": inlier_ratio,
            "slope_percent": slope_info["slope_percent"],
            "slope_angle_deg": slope_info["slope_angle_deg"],
            "status": "ok"
        })

    df = pd.DataFrame(rows)
    out_csv = os.path.join(dataset_dir, "phase1_slope_labels.csv")
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"[DONE] saved: {out_csv}")
    print(df.head(10))
    return df


if __name__ == "__main__":
    # 예: python phase1_extract_slope.py
    dataset_dir = "datas/Depth_001"
    run_phase1(dataset_dir)

Found conf: ['Depth_001.conf', 'Depth_002.conf', 'Depth_003.conf', 'Depth_004.conf', 'Depth_005.conf']
